In [ ]:
1+3

In [ ]:
# %matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
%matplotlib widget

In [ ]:
# Load the CSV file
csv_filename = "v_meas_2025-12-24 15:53:14_2025-12-25 01:41:38.csv"
df = pd.read_csv(csv_filename)

concat_time = df['time_s'].values
concat_voltage = df['voltage_v'].values

print(f"Loaded {len(concat_time)} data points")
print(f"Time range: {concat_time[0]:.2f} s to {concat_time[-1]:.2f} s")
print(f"Total duration: {concat_time[-1] - concat_time[0]:.2f} s")
print(f"Voltage range: {np.min(concat_voltage):.6f} V to {np.max(concat_voltage):.6f} V")

In [ ]:
# Select a specific range in the data
# You can modify these indices to select different ranges
start_idx = int(52640000*0)  # Start index (0 = beginning)
end_idx = len(concat_time)  # End index (len = end of data)

# Or select by time (in seconds)
# start_time = 0.0  # Start time in seconds
# end_time = 100.0  # End time in seconds
# start_idx = np.argmin(np.abs(concat_time - start_time))
# end_idx = np.argmin(np.abs(concat_time - end_time))

# Extract the selected range
time_selected = concat_time[start_idx:end_idx]
voltage_selected = concat_voltage[start_idx:end_idx]

print(f"Selected range: indices {start_idx} to {end_idx}")
print(f"Selected time range: {time_selected[0]:.2f} s to {time_selected[-1]:.2f} s")
print(f"Number of points: {len(time_selected)}")
print(f"Duration: {time_selected[-1] - time_selected[0]:.2f} s")

In [ ]:
# Remove the mean from the data
voltage_mean = np.mean(voltage_selected)
voltage_demeaned = voltage_selected - voltage_mean

print(f"Original mean: {voltage_mean:.6f} V")
print(f"New mean (should be ~0): {np.mean(voltage_demeaned):.6f} V")
print(f"Original std: {np.std(voltage_selected):.6f} V")
print(f"New std: {np.std(voltage_demeaned):.6f} V")

In [ ]:
# Apply Hamming window
hamming_window = np.hamming(len(voltage_demeaned))
voltage_windowed = voltage_demeaned * hamming_window

# Plot the original, demeaned, and windowed signals for comparison
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Original signal
axes[0].plot(time_selected, voltage_selected, linewidth=0.5)
axes[0].set_ylabel("Voltage (V)", fontsize=12)
axes[0].set_title("Original Signal (Selected Range)", fontsize=14)
axes[0].grid(True, alpha=0.3)

# Demeaned signal
axes[1].plot(time_selected, voltage_demeaned, linewidth=0.5, color='orange')
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=1, alpha=0.5)
axes[1].set_ylabel("Voltage (V)", fontsize=12)
axes[1].set_title("Demeaned Signal (Mean Removed)", fontsize=14)
axes[1].grid(True, alpha=0.3)

# Windowed signal
axes[2].plot(time_selected, voltage_windowed, linewidth=0.5, color='green')
axes[2].axhline(y=0, color='r', linestyle='--', linewidth=1, alpha=0.5)
axes[2].set_xlabel("Time (s)", fontsize=12)
axes[2].set_ylabel("Voltage (V)", fontsize=12)
axes[2].set_title("Windowed Signal (Hamming Window Applied)", fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(csv_filename.replace('.csv', '_1.png'))
plt.show()

In [ ]:
# Calculate sampling frequency and perform FFT
dt = np.mean(np.diff(time_selected))
sampling_freq = 1.0 / dt if dt > 0 else 1.0

# Perform FFT on the windowed signal
fft_voltage = np.fft.fft(voltage_windowed)
fft_freq = np.fft.fftfreq(len(voltage_windowed), dt)

# Get only positive frequencies
positive_freq_idx = fft_freq > 0
positive_freq = fft_freq[positive_freq_idx]
magnitude = np.abs(fft_voltage[positive_freq_idx])

# Plot the frequency spectrum
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Linear scale
axes[0].plot(positive_freq, magnitude, linewidth=1)
axes[0].set_xlabel("Frequency (Hz)", fontsize=12)
axes[0].set_ylabel("Magnitude", fontsize=12)
axes[0].set_title("FFT of Windowed Signal (Linear Scale)", fontsize=14)
axes[0].grid(True, alpha=0.3)

# Log-log scale
axes[1].plot(positive_freq, magnitude, linewidth=1)
axes[1].set_xlabel("Frequency (Hz)", fontsize=12)
axes[1].set_ylabel("Magnitude", fontsize=12)
axes[1].set_title("FFT of Windowed Signal (Log-Log Scale)", fontsize=14)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(csv_filename.replace('.csv', '_2.png'))
plt.show()

# Print frequency domain statistics
print("\nFrequency domain stats:")
print(f"  Sampling frequency: {sampling_freq:.2f} Hz")
print(f"  Nyquist frequency: {sampling_freq/2:.2f} Hz")
print(f"  Frequency resolution: {positive_freq[1] - positive_freq[0]:.6f} Hz")
print(f"  Dominant frequency: {positive_freq[np.argmax(magnitude)]:.6f} Hz")
print(f"  Max magnitude: {np.max(magnitude):.6f}")
print(f"  Number of frequency bins: {len(positive_freq)}")